# Kaggle setup
In Kaggle, choose **GPU** (T4) and turn **Internet on**. The repo's training script is used unmodified (the earlier patch cell was removed because its text patterns did not match the real source).

In [ ]:
import torch, os
from pathlib import Path
print('CUDA:', torch.cuda.is_available(), 'GPU count:', torch.cuda.device_count(), 'CPU cores:', os.cpu_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()): print(i, torch.cuda.get_device_name(i))
if not Path('Deep-Learning-Project').exists():
    !git clone https://github.com/saishthakur/Deep-Learning-Project
%cd Deep-Learning-Project
!pip install -q pandas scikit-learn transformers numpy joblib matplotlib kagglehub tqdm
!mkdir -p data


## Download the dataset

Downloads the ISOT Fake News Dataset from its public Kaggle dataset page and extracts the archive. Kaggle Internet must be enabled.

In [ ]:
import kagglehub, shutil, os
from pathlib import Path
archive_dir = Path(kagglehub.dataset_download('clmentbisaillon/fake-and-real-news-dataset'))
print('Downloaded dataset to:', archive_dir)
matches = list(archive_dir.rglob('Fake.csv')) + list(archive_dir.rglob('True.csv'))
found = {p.name: p for p in matches}
required = {'Fake.csv', 'True.csv'}
missing = required - set(found)
if missing: raise FileNotFoundError(f'Dataset download did not contain: {sorted(missing)}')
for name in sorted(required): shutil.copy2(found[name], Path('data') / name)
print('Dataset files:', [(name, (Path('data')/name).stat().st_size) for name in sorted(required)])

## Shared split

Prepares the deduplicated stratified train, validation, and test data.

In [ ]:
!python -m src.split_data --data-dir data --output-dir data/processed --seed 42
!zip -rq split_checkpoint.zip data/processed
from IPython.display import FileLink, display
display(FileLink('split_checkpoint.zip'))

In [ ]:
from pathlib import Path

p = Path("src/train_model.py")
s = p.read_text()

if "import os\n" not in s:
    s = s.replace("import argparse\n", "import argparse\nimport os\n", 1)

compile(s, str(p), "exec")
p.write_text(s)
print("Added the missing os import; training script syntax is valid.")

## Train, evaluate and compare

With two GPUs, one worker trains/evaluates DistilBERT on physical GPU 0 while another runs MLP, CNN, and LSTM sequentially on physical GPU 1. With one GPU, all models run sequentially. Each model writes separate artifacts and logs; the workers stop with a clear error if any command fails. Shortened settings: 5 epochs for MLP/CNN/LSTM, 2 epochs and length 128 for DistilBERT.

In [ ]:
import os, sys, time, subprocess, threading
from pathlib import Path
import torch
n_gpu = torch.cuda.device_count()
if n_gpu < 1: raise RuntimeError('Enable a GPU accelerator in Kaggle settings.')
args = {'mlp':['--model','mlp','--epochs','5'], 'cnn':['--model','cnn','--epochs','5'], 'lstm':['--model','lstm','--epochs','5'], 'distilbert':['--model','distilbert','--epochs','2','--max-length','128','--batch-size','32']}
queues = {0:['distilbert'], 1:['mlp','cnn','lstm']} if n_gpu >= 2 else {0:['mlp','cnn','lstm','distilbert']}
print('GPU count:', n_gpu, 'model queues:', queues, flush=True)
Path('logs').mkdir(exist_ok=True)
def run(cmd, gpu, tag):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    log = Path('logs') / f'{tag}.log'
    with log.open('w', encoding='utf-8') as f:
        p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout: print(f'[GPU {gpu} {tag}] {line}', end='', flush=True); f.write(line); f.flush()
        code = p.wait()
    if code: raise RuntimeError(f'{tag} failed (exit {code}); full log: {log}')
def worker(gpu, names, errors):
    try:
        for name in names:
            out = Path('artifacts') / name
            ckpt = out / ('transformer/config.json' if name == 'distilbert' else 'best_model.pt')
            done = (out / 'run_config.json').exists() and ckpt.exists()
            if not done:
                run([sys.executable,'-u','-m','src.train_model',*args[name]], gpu, f'train_{name}')
            else: print(f'[GPU {gpu}] training checkpoint exists for {name}; evaluating it', flush=True)
            run([sys.executable,'-u','-m','src.evaluate_model','--model',name], gpu, f'eval_{name}')
    except Exception as e: errors.append(e)
errors=[]
workers=[threading.Thread(target=worker,args=(gpu,names,errors)) for gpu,names in queues.items()]
for t in workers: t.start()
for t in workers: t.join()
if errors: raise RuntimeError('; '.join(map(str,errors)))
run([sys.executable,'-u','-m','src.compare_models','--artifact-root','artifacts','--output-dir','results'],0,'compare')
print('All models trained, evaluated, and compared.')

In [ ]:
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv('results/model_comparison.csv'))
display(Image('results/training_and_f1.png'))
display(Image('results/confusion_matrices.png'))

## Review errors and summarize

Inspect the misclassified examples before writing conclusions. The next cell shows up to 10 examples per model. Assign error categories only after human review, and do not infer that dataset performance proves real-world truth detection.

In [ ]:
import json
from pathlib import Path
for model_name in ['mlp', 'cnn', 'lstm', 'distilbert']:
    metrics = json.loads(Path(f'artifacts/{model_name}/test_metrics.json').read_text())
    errors = metrics['misclassified_articles']
    print(f'\\n{model_name}: {len(errors)} errors; showing up to 10')
    for err in errors[:10]: print({k: err[k] for k in ('id','true_label','predicted_label','text_excerpt')})
print('\\nProject summary table:')
display(pd.read_csv('results/model_comparison.csv'))

## Run repository checks

These synthetic tests do not require the ISOT dataset.

In [ ]:
!python -m unittest discover -s tests -v

## Download results

Downloads the report, error-analysis artifacts, and split manifest. Model weights are excluded to keep the bundle smaller.

In [ ]:
!zip -rq results_bundle.zip results data/processed artifacts -x 'artifacts/*/*.pt' 'artifacts/*/transformer/*'
from IPython.display import FileLink, display
display(FileLink('results_bundle.zip'))

In [ ]:
from pathlib import Path
from IPython.display import display, FileLink
from IPython.utils import io

# Include results, logs, split files, model artifacts, and checkpoints.
# Skip the original downloaded CSVs to keep the ZIP smaller.
bundle = Path("/kaggle/working/results_bundle_full.zip")

!zip -rq "{bundle}" results artifacts data/processed logs

print(f"Bundle created: {bundle}")
print(f"Size: {bundle.stat().st_size / (1024**3):.2f} GB")
display(FileLink(str(bundle)))

In [ ]:
from pathlib import Path
from IPython.display import FileLink, display
import zipfile

bundle = Path("results_bundle_full.zip")

with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for folder in ("results", "artifacts", "data/processed", "logs"):
        path = Path(folder)
        if path.exists():
            for file in path.rglob("*"):
                if file.is_file():
                    z.write(file)

print("Created:", bundle.resolve())
print("Size:", round(bundle.stat().st_size / (1024**3), 2), "GB")
display(FileLink("results_bundle_full.zip"))

In [ ]:
!zip -rq /kaggle/working/results_bundle_full.zip \
    results artifacts data logs